In [ ]:
%cd /content
!rm -rf autoresearch
!git clone -b integrate-preflight-governor https://github.com/bh3r1th/autoresearch.git
%cd /content/autoresearch
!git remote -v
!git branch --show-current
!git rev-parse HEAD

/content
Cloning into 'autoresearch'...
remote: Enumerating objects: 196, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 196 (delta 0), reused 1 (delta 0), pack-reused 193 (from 1)
Receiving objects: 100% (196/196), 545.14 KiB | 28.69 MiB/s, done.
Resolving deltas: 100% (98/98), done.
/content/autoresearch
origin	https://github.com/bh3r1th/autoresearch.git (fetch)
origin	https://github.com/bh3r1th/autoresearch.git (push)
integrate-preflight-governor
6c9626cec5c290c8b1b5bbc8167a570f9d089a3e


In [ ]:
!grep -n "AUTORESEARCH_GOVERNOR" train.py
!grep -n "hooks" train.py
!grep -n "record" train.py

36:GOVERNOR_ENABLED = os.environ.get("AUTORESEARCH_GOVERNOR_ENABLED", "1").lower() not in {"0", "false", "no", "off"}
37:GOVERNOR_HISTORY_PATH = os.environ.get("AUTORESEARCH_GOVERNOR_HISTORY_PATH", ".autoresearch/governor_history.jsonl")
38:GOVERNOR_RUN_ID = os.environ.get("AUTORESEARCH_GOVERNOR_RUN_ID") or f"{int(time.time())}-{os.getpid()}"
88:            from preflight_intent_governor.hooks import record_execution_result, run_preflight_check
96:            from preflight_intent_governor.hooks import record_execution_result, run_preflight_check
87:            from preflight_intent_governor.history import load_attempt_records
88:            from preflight_intent_governor.hooks import record_execution_result, run_preflight_check
95:            from preflight_intent_governor.history import load_attempt_records
96:            from preflight_intent_governor.hooks import record_execution_result, run_preflight_check
102:            "load_attempt_records": load_attempt_records,
104:         

In [ ]:
!uv sync
!uv run prepare.py --num-shards 8

Resolved 74 packages in 1ms
Checked 67 packages in 0.98ms
Cache directory: /root/.cache/autoresearch

Data: all 9 shards already downloaded at /root/.cache/autoresearch/data

Tokenizer: already trained at /root/.cache/autoresearch/tokenizer

Done! Ready to train.


In [ ]:
# with eps=1e-10 in train.py
!AUTORESEARCH_GOVERNOR_ENABLED=1 \
 AUTORESEARCH_GOVERNOR_HISTORY_PATH=/content/governor.jsonl \
 AUTORESEARCH_GOVERNOR_RUN_ID=smoke-1 \
 uv run train.py

[governor] block_id=blk_59a25720b9e994f76459
[governor] extracted_params=11
[governor] preflight_decision=allow_no_conflict allowed=True
Vocab size: 8,192
Model config: {'sequence_len': 2048, 'vocab_size': 8192, 'n_layer': 8, 'n_head': 4, 'n_kv_head': 4, 'n_embd': 512, 'window_pattern': 'SSSL'}
Parameter counts:
  wte                     : 4,194,304
  value_embeds            : 16,777,216
  lm_head                 : 4,194,304
  transformer_matrices    : 25,166,336
  scalars                 : 16
  total                   : 50,332,176
Estimated FLOPs per token: 2.390784e+08
Scaling AdamW LRs by 1/sqrt(512/768) = 1.224745
Time budget: 300s
Gradient accumulation steps: 2
step 00384 (99.7%) | loss: 3.057002 | lrm: 0.01 | dt: 802ms | tok/sec: 653,876 | mfu: 15.8% | epoch: 1 | remaining: 0s    
---
val_bpb:          1.090975
training_seconds: 300.0
total_seconds:    352.9
peak_vram_mb:     45012.5
mfu_percent:      15.83
total_tokens_M:   201.9
num_steps:        385
num_params_M:     50.3
dept

In [ ]:
!ls -l /content/governor.jsonl
!tail -20 /content/governor.jsonl

-rw-r--r-- 1 root root 3980 Apr  8 04:12 /content/governor.jsonl
{"attempt_id":"att_a2d86ea2544dc56c4024cdb4","block_id":"blk_59a25720b9e994f76459","failure_tags":[],"file_path":"/content/autoresearch/train.py","function_name":"setup_optimizer","line_number":409,"outcome":"success","param_name":"weight_decay","param_value_normalized":"0.0000000000","param_value_raw":"0.0","run_id":"smoke-1","timestamp":"2026-04-08T04:12:43Z"}
{"attempt_id":"att_70d986123a658c07f21f7dc6","block_id":"blk_59a25720b9e994f76459","failure_tags":[],"file_path":"/content/autoresearch/train.py","function_name":"setup_optimizer","line_number":423,"outcome":"success","param_name":"eps","param_value_normalized":"0.0000000001","param_value_raw":"1e-10","run_id":"smoke-1","timestamp":"2026-04-08T04:12:43Z"}
{"attempt_id":"att_0f596ebc52e7555f17646ec5","block_id":"blk_59a25720b9e994f76459","failure_tags":[],"file_path":"/content/autoresearch/train.py","function_name":"setup_optimizer","line_number":423,"outcome":"suc

In [ ]:
!python - <<'PY'
import json
rows = [json.loads(x) for x in open("/content/governor.jsonl")]
print(sorted({r["param_name"] for r in rows}))

/bin/bash: line 1: warning: here-document at line 1 delimited by end-of-file (wanted `PY')
['eps', 'weight_decay']


In [ ]:
# with eps=-1e-10 in train.py
!AUTORESEARCH_GOVERNOR_ENABLED=1 \
 AUTORESEARCH_GOVERNOR_HISTORY_PATH=/content/governor.jsonl \
 AUTORESEARCH_GOVERNOR_RUN_ID=fail-1 \
 uv run train.py

[governor] block_id=blk_59a25720b9e994f76459
[governor] extracted_params=11
[governor] preflight_decision=allow_no_conflict allowed=True
Vocab size: 8,192
Model config: {'sequence_len': 2048, 'vocab_size': 8192, 'n_layer': 8, 'n_head': 4, 'n_kv_head': 4, 'n_embd': 512, 'window_pattern': 'SSSL'}
Parameter counts:
  wte                     : 4,194,304
  value_embeds            : 16,777,216
  lm_head                 : 4,194,304
  transformer_matrices    : 25,166,336
  scalars                 : 16
  total                   : 50,332,176
Estimated FLOPs per token: 2.390784e+08
Scaling AdamW LRs by 1/sqrt(512/768) = 1.224745
Time budget: 300s
Gradient accumulation steps: 2
step 00040 (7.7%) | loss: 5.163802 | lrm: 1.00 | dt: 798ms | tok/sec: 656,766 | mfu: 15.9% | epoch: 1 | remaining: 276s    FAIL
[governor] block_id=blk_59a25720b9e994f76459
[governor] extracted_params=11
[governor] failure_tags=['runtime_error']
[governor] history_write_count=11


In [ ]:
# with eps=-1e-10 in train.py
!AUTORESEARCH_GOVERNOR_ENABLED=1 \
 AUTORESEARCH_GOVERNOR_HISTORY_PATH=/content/governor.jsonl \
 AUTORESEARCH_GOVERNOR_RUN_ID=fail-2 \
 uv run train.py

[governor] block_id=blk_59a25720b9e994f76459
[governor] extracted_params=11
[governor] preflight_decision=block_repeated_failed_value allowed=False
[governor] blocked run for block_id=blk_59a25720b9e994f76459: block_repeated_failed_value
[governor] matched_attempt_ids=['att_24bb8106fdebaa9963caf06a', 'att_84b114cada04ac8b2bec9871', 'att_fe997524e2eb2f88f8a9b8ee', 'att_f2d16d0b9e4e1c4ab7aa6da8', 'att_e328dea51582509697bf3703', 'att_af44af2b50778ea50b6e2df6', 'att_f804064bf9ca6b727af8ea57', 'att_b0066a3803779d081fd19145', 'att_1615d3de32422b61774829e7', 'att_5bdb0b13ae616ae22696acb1', 'att_2a4aa7846044e6c9d385d8e3', 'att_24bb8106fdebaa9963caf06a', 'att_84b114cada04ac8b2bec9871', 'att_fe997524e2eb2f88f8a9b8ee', 'att_f2d16d0b9e4e1c4ab7aa6da8', 'att_e328dea51582509697bf3703', 'att_af44af2b50778ea50b6e2df6', 'att_f804064bf9ca6b727af8ea57', 'att_b0066a3803779d081fd19145', 'att_1615d3de32422b61774829e7', 'att_5bdb0b13ae616ae22696acb1', 'att_2a4aa7846044e6c9d385d8e3', 'att_24bb8106fdebaa9963caf

In [ ]:
# with eps=1e-10 in train.py
!AUTORESEARCH_GOVERNOR_ENABLED=1 \
 AUTORESEARCH_GOVERNOR_HISTORY_PATH=/content/governor.jsonl \
 AUTORESEARCH_GOVERNOR_RUN_ID=pass-1 \
 uv run train.py

[governor] block_id=blk_59a25720b9e994f76459
[governor] extracted_params=11
[governor] preflight_decision=allow_new_direction allowed=True
Vocab size: 8,192
Model config: {'sequence_len': 2048, 'vocab_size': 8192, 'n_layer': 8, 'n_head': 4, 'n_kv_head': 4, 'n_embd': 512, 'window_pattern': 'SSSL'}
Parameter counts:
  wte                     : 4,194,304
  value_embeds            : 16,777,216
  lm_head                 : 4,194,304
  transformer_matrices    : 25,166,336
  scalars                 : 16
  total                   : 50,332,176
Estimated FLOPs per token: 2.390784e+08
Scaling AdamW LRs by 1/sqrt(512/768) = 1.224745
Time budget: 300s
Gradient accumulation steps: 2
step 00384 (99.8%) | loss: 3.051080 | lrm: 0.00 | dt: 801ms | tok/sec: 654,257 | mfu: 15.8% | epoch: 1 | remaining: 0s    
---
val_bpb:          1.087593
training_seconds: 300.3
total_seconds:    352.5
peak_vram_mb:     45012.5
mfu_percent:      15.82
total_tokens_M:   201.9
num_steps:        385
num_params_M:     50.3
de